# 課題5 Word2Vec

単語をベクトルで表現し、`king - man + woman` のような意味的な演算を試します。

まず研究室共通データセット内のローカル word vector を探します。見つからない場合だけ、最後の fallback として gensim downloader を使えるようにしています。

In [1]:
from pathlib import Path
import os
import numpy as np
import scipy.linalg

# gensim import error 対策
if not hasattr(scipy.linalg, "triu"):
    scipy.linalg.triu = np.triu

In [2]:
import gensim.downloader as api
from gensim.models import KeyedVectors

# JupyterHub上での課題フォルダ。結果テキストは課題4のout/に保存する。
BASE_DIR = Path('/export/space0/oyundari/jupyter/notebook/kadai4/dnn_kadai2')
TASK_DIR = BASE_DIR / 'kadai5-word2vec'
OUT_DIR = TASK_DIR / 'out'
MODEL_DIR = TASK_DIR / 'model'
# 研究室共通データセットからword vectorを探し、新規ダウンロードを避ける。
LAB_DATA_ROOT = Path('/export/data/dataset')
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

## モデル読み込み

`/export/data/dataset/google` などに GoogleNews word2vec がある場合はそれを使います。環境にローカルモデルがない場合は、`ALLOW_DOWNLOAD=True` にした時だけ軽量な GloVe を取得します。

In [3]:
# 学内ネットワーク経由の場合はプロキシを設定
os.environ["http_proxy"]  = "http://proxy.uec.ac.jp:8080/"
os.environ["https_proxy"] = "http://proxy.uec.ac.jp:8080/"

# IPv6 を無効化（接続問題の回避）
os.environ["HF_HUB_DISABLE_IPV6"] = "1"

In [5]:
# Trueにするとgensimから小さいGloVeを取得する。通常はFalseのままローカルデータを使う。
ALLOW_DOWNLOAD = True

# よく置かれているGoogleNews word2vecの候補パスを順番に確認する。
def find_local_word_vector():
    candidates = [
        LAB_DATA_ROOT / 'google' / 'GoogleNews-vectors-negative300.bin',
        LAB_DATA_ROOT / 'google' / 'GoogleNews-vectors-negative300.bin.gz',
        LAB_DATA_ROOT / 'GoogleNews-vectors-negative300.bin',
        LAB_DATA_ROOT / 'GoogleNews-vectors-negative300.bin.gz',
    ]
    for path in candidates:
        if path.exists():
            return path
    return None

local_model = find_local_word_vector()
if local_model is not None:
    print('loading local model:', local_model)
    # メモリ使用量を抑えるため、先頭30万語まで読み込む。
    wv = KeyedVectors.load_word2vec_format(str(local_model), binary=True, limit=300000)
elif ALLOW_DOWNLOAD:
    print('local model was not found. downloading small GloVe model via gensim...')
    wv = api.load('glove-wiki-gigaword-50')
else:
    raise FileNotFoundError('Local word vector was not found. Put GoogleNews vectors under /export/data/dataset/google or set ALLOW_DOWNLOAD=True.')

print('vector size:', wv.vector_size)

local model was not found. downloading small GloVe model via gensim...
[=================---------------------------------] 35.5% 23.4/66.0MB downloaded

IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)



[==================================================] 100.0% 66.0/66.0MB downloaded
vector size: 50


## 単語演算

`most_similar(positive=..., negative=...)` は、足し算したい単語を `positive`、引き算したい単語を `negative` に入れます。結果として、演算後のベクトルに近い単語が返されます。

In [7]:
# positiveは足し算する単語、negativeは引き算する単語。
tests = [
    (['king', 'woman'], ['man']),
    (['paris', 'japan'], ['france']),
    (['tokyo', 'france'], ['japan']),
    (['walking', 'swim'], ['walk']),
    (['phone', 'video'], ['call']),
    (['doctor', 'woman'], ['man']),
]

lines = []
for positive, negative in tests:
    header = f'positive={positive}, negative={negative}'
    print('\n' + header)
    lines.append(header)
    # 演算後のベクトルに近い単語を上位5件表示する。
    for word, score in wv.most_similar(positive=positive, negative=negative, topn=5):
        row = f'{word}\t{score:.3f}'
        print(row)
        lines.append(row)

(OUT_DIR / 'word2vec_results.txt').write_text('\n'.join(lines), encoding='utf-8')


positive=['king', 'woman'], negative=['man']
queen	0.852
throne	0.766
prince	0.759
daughter	0.747
elizabeth	0.746

positive=['paris', 'japan'], negative=['france']
tokyo	0.923
shanghai	0.837
japanese	0.790
osaka	0.783
beijing	0.748

positive=['tokyo', 'france'], negative=['japan']
paris	0.921
brussels	0.795
strasbourg	0.790
prohertrib	0.780
french	0.742

positive=['walking', 'swim'], negative=['walk']
swimming	0.807
swimmers	0.750
swims	0.745
surfing	0.741
biking	0.736

positive=['phone', 'video'], negative=['call']
audio	0.776
digital	0.764
clips	0.763
videos	0.758
youtube	0.752

positive=['doctor', 'woman'], negative=['man']
nurse	0.840
child	0.766
pregnant	0.757
mother	0.752
patient	0.752


695